### **Installs and Imports**

In [ ]:
!pip install -q transformers datasets peft

# Upgrade torchao to a compatible version
!pip install --upgrade torchao

import torch
import requests
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 101.1 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


### **Load the base model** and tokenizer, put the model on the GPU.

In [ ]:
model_id = "HuggingFaceTB/SmolLM-135M"     # BASE, not -Instruct

device    = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = AutoTokenizer.from_pretrained(model_id)
model     = AutoModelForCausalLM.from_pretrained(model_id).to(device)

print(f"Loaded {model_id} on {device} — {sum(p.numel() for p in model.parameters()):,} params")

config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.69k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  538MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Loaded HuggingFaceTB/SmolLM-135M on cuda — 134,515,008 params


### **Wrap the Model with LoRA Adapters:**

In [ ]:
lora_config = LoraConfig(
    task_type      = "CAUSAL_LM",              # tells peft this is a next-token LM
    r              = 8,                          # the rank — size of A and B
    lora_alpha     = 16,                         # scaling: effective ΔW = (alpha/r)·B·A
    target_modules = ["q_proj", "v_proj"],       # WHICH layers get adapters
    lora_dropout   = 0.05,                        # dropout on the adapter path
    bias           = "none",                      # don't train bias terms
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 460,800 || all params: 134,975,808 || trainable%: 0.3414


### **Save the Model:**

In [ ]:
model.save_pretrained("smollm-lora-shakespeare")   # saves ONLY the adapters — a few MB, not 500MB

### **Reload and add the adapters:**

In [ ]:
from peft import PeftModel
base  = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM-135M").to(device)
model = PeftModel.from_pretrained(base, "smollm-lora-shakespeare").to(device)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]